# Etapa 2: Feature Engineering de Alto Rendimiento

**Objetivo de Negocio (BPO Financiero):**
Un modelo no detecta fraudes mirando una transacción aislada, necesita contexto. Construiremos variables de comportamiento temporal (ej: hora del día, medias móviles de gasto) para darle ese contexto al algoritmo.

**Stack Tecnológico:**
Utilizaremos **DuckDB** para procesar el CSV original mediante consultas SQL analíticas (Window Functions), optimizando el uso de memoria y reduciendo el tiempo de procesamiento antes de pasar los datos a Pandas.

In [1]:
import duckdb
import pandas as pd
import time

# Conectar a una base de datos DuckDB en memoria (ultrarrápida)
con = duckdb.connect(database=':memory:')
print("Motor DuckDB inicializado correctamente.")

Motor DuckDB inicializado correctamente.


In [2]:
# Medimos el rendimiento
start_time = time.time()

# La tubería: Leemos directo el CSV, creamos features y devolvemos un DataFrame
query = """
WITH base_data AS (
    SELECT 
        *,
        -- 1. Transformar segundos transcurridos a 'Hora del dia' (ciclo de 24h)
        CAST(FLOOR((Time / 3600) % 24) AS INTEGER) AS hour_of_day
    FROM read_csv_auto('../data/raw/creditcard.csv')
),
feature_data AS (
    SELECT 
        *,
        -- 2. Media móvil del monto de las últimas 50 transacciones
        AVG(Amount) OVER (
            ORDER BY Time 
            ROWS BETWEEN 50 PRECEDING AND CURRENT ROW
        ) AS rolling_avg_amount_50,
        
        -- 3. Que tan lejos esta la transaccion actual del promedio reciente
        Amount - AVG(Amount) OVER (
            ORDER BY Time 
            ROWS BETWEEN 50 PRECEDING AND CURRENT ROW
        ) AS amount_diff_from_avg
    FROM base_data
)
SELECT * FROM feature_data;
"""

df_features = con.execute(query).df()

ejecucion = time.time() - start_time
print(f"Tubería de Feature Engineering completada en {ejecucion:.2f} segundos.")
print(f"Nuevas dimensiones del dataset: {df_features.shape}")

Tubería de Feature Engineering completada en 1.26 segundos.
Nuevas dimensiones del dataset: (284807, 34)


In [3]:
# Revisar las cañerías para ver las nuevas variables inyectadas
columnas_clave = ['Time', 'hour_of_day', 'Amount', 'rolling_avg_amount_50', 'amount_diff_from_avg', 'Class']
df_features[columnas_clave].tail(10)

,Time,hour_of_day,Amount,rolling_avg_amount_50,amount_diff_from_avg,Class
284797,172782,23,5.49,37.015294,-31.525294,0
284798,172782,23,24.05,37.302157,-13.252157,0
284799,172783,23,79.99,34.551373,45.438627,0
284800,172784,23,2.68,34.449412,-31.769412,0
284801,172785,23,2.69,34.476863,-31.786863,0
284802,172786,23,0.77,34.474510,-33.704510,0
284803,172787,23,24.79,34.470392,-9.680392,0
284804,172788,23,67.88,29.182941,38.697059,0
284805,172788,23,10.00,26.320196,-16.320196,0
284806,172792,23,217.00,30.379216,186.620784,0


In [ ]:
# Guardar el dataset con las nuevas variables en formato Parquet (Alta eficiencia)
import os
os.makedirs('../data/features', exist_ok=True)

df_features.to_parquet('../data/features/temporal_features_v1.parquet', index=False)
print("Features guardadas exitosamente en data/features/temporal_features_v1.parquet")